# 02 - Feature Engineering | B3 Quant Analytics

Notebook para demonstrar pipeline técnico: limpeza, transformações, SQL avançado (CTE + window function) e criação de features de risco.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from utils.config_loader import load_settings
from ingestion.fetch_db import initialize_database, load_returns_with_sql
from ingestion.fetch_api import fetch_market_data
from processing.clean import clean_prices
from processing.transform import build_returns_frame
from features.build_features import build_risk_return_features

settings = load_settings(PROJECT_ROOT)

In [2]:
raw_df = fetch_market_data(
    symbols=settings.symbols,
    start_date=settings.start_date,
    end_date=settings.end_date,
    interval=settings.interval,
)
clean_df = clean_prices(raw_df)
returns_df = build_returns_frame(clean_df)

print("Raw shape:", raw_df.shape)
print("Clean shape:", clean_df.shape)
print("Returns shape:", returns_df.shape)
returns_df.head()

Baixando ativos: 100%|██████████| 4/4 [00:02<00:00,  1.51ativo/s]

Raw shape: (5976, 8)
Clean shape: (5976, 8)
Returns shape: (5976, 10)


,date,adj_close,close,high,low,open,volume,ticker,daily_return,log_return
0,2020-01-02,19.295868,28.181818,28.181818,27.197596,27.445530,25031584.0,BBDC4.SA,NaN,NaN
1,2020-01-03,19.305622,28.181818,28.549961,27.708488,27.723516,39999078.0,BBDC4.SA,0.000000,0.000000
2,2020-01-06,18.960787,27.678436,28.009014,27.377911,27.948910,33675497.0,BBDC4.SA,-0.017862,-0.018023
3,2020-01-07,18.631399,27.197596,27.768595,27.047333,27.610819,19775066.0,BBDC4.SA,-0.017372,-0.017525
4,2020-01-08,18.343174,26.776859,27.438017,26.641623,27.325319,28306110.0,BBDC4.SA,-0.015470,-0.015591


In [3]:
memory_before = raw_df.memory_usage(deep=True).sum() / 1024**2
memory_after = clean_df.memory_usage(deep=True).sum() / 1024**2

missing_table = returns_df.isna().mean().sort_values(ascending=False).to_frame("missing_ratio")

pd.DataFrame(
    {
        "memoria_raw_mb": [round(memory_before, 3)],
        "memoria_clean_mb": [round(memory_after, 3)],
        "redução_percentual": [round((1 - memory_after / memory_before) * 100, 2)],
    }
)

,memoria_raw_mb,memoria_clean_mb,redução_percentual
0,0.644,0.211,67.18


In [4]:
features_df = build_risk_return_features(
    df=returns_df,
    rolling_window=settings.rolling_window,
    risk_free_rate_annual=settings.risk_free_rate_annual,
    confidence_level=settings.confidence_level,
)

feature_cols = [
    "daily_return",
    "log_return",
    "volatility_rolling",
    "var_historical_rolling",
    "sharpe_ratio",
    "adf_pvalue",
]
features_df[feature_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
daily_return,5972.0,2.720252e-04,0.023158,-0.296978,-0.010395,0.000000,0.011148,0.222222
log_return,5972.0,5.125331e-07,0.023400,-0.352367,-0.010449,0.000000,0.011087,0.200671
volatility_rolling,5892.0,3.213317e-01,0.183419,0.060744,0.219129,0.283652,0.365320,1.936495
var_historical_rolling,5892.0,-2.720102e-02,0.018796,-0.205047,-0.032931,-0.023529,-0.016792,-0.002994
sharpe_ratio,5976.0,-1.519077e-01,0.130650,-0.362412,-0.209172,-0.108229,-0.050965,-0.028760


In [5]:
db_path = initialize_database(settings.database_url, settings.root_dir)
sql_returns = load_returns_with_sql(db_path)

compare = (
    returns_df[["date", "ticker", "daily_return"]]
    .merge(
        sql_returns[["date", "ticker", "daily_return"]],
        on=["date", "ticker"],
        suffixes=("_pandas", "_sql"),
    )
    .dropna()
)

compare["abs_diff"] = (compare["daily_return_pandas"] - compare["daily_return_sql"]).abs()
compare[["daily_return_pandas", "daily_return_sql", "abs_diff"]].describe()

,daily_return_pandas,daily_return_sql,abs_diff
count,5972.000000,5972.000000,5.972000e+03
mean,0.000272,0.000272,2.235321e-08
std,0.023158,0.023158,1.557855e-08
min,-0.296978,-0.296978,0.000000e+00
25%,-0.010395,-0.010395,1.004353e-08
50%,0.000000,0.000000,1.953032e-08
75%,0.011148,0.011148,3.047879e-08
max,0.222222,0.222222,5.958878e-08


In [6]:
out_path = PROJECT_ROOT / "data" / "processed" / "features_interview_demo.csv"
features_df.to_csv(out_path, index=False)

## Tópicos de discussão
- Demonstro rastreabilidade da feature: da API até o banco relacional.
- Comparo retorno calculado em Pandas vs SQL (CTE + window) para validar consistência.
- Mostro preocupação com eficiência (memória e qualidade de dados) antes da modelagem.